In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('SuperMarket - SuperMarket.csv')

In [3]:
df.shape

(1014, 17)

In [4]:
df.dtypes

Invoice ID                     str
Branch                         str
City                           str
Customer type                  str
Gender                         str
Product line                   str
Unit price                 float64
Quantity                     int64
Tax 5%                     float64
Sales                          str
Date                           str
Time                           str
Payment                        str
cogs                       float64
gross margin percentage    float64
gross income               float64
Rating                     float64
dtype: object

In [5]:
df.isnull().sum()

Invoice ID                  0
Branch                      0
City                        0
Customer type               0
Gender                      0
Product line                0
Unit price                 24
Quantity                    0
Tax 5%                      0
Sales                      20
Date                       12
Time                        0
Payment                     0
cogs                        0
gross margin percentage     0
gross income                0
Rating                      0
dtype: int64

In [6]:
df.duplicated().sum()

np.int64(14)

In [7]:
df.describe()

,Unit price,Quantity,Tax 5%,cogs,gross margin percentage,gross income,Rating
count,990.000000,1014.000000,1014.000000,1014.000000,1014.000000,1014.000000,1014.000000
mean,55.520333,5.502959,15.337247,306.737436,4.761905,15.337247,6.960454
std,26.330508,2.927133,11.685745,233.718896,0.000000,11.685745,1.721264
min,10.080000,1.000000,0.508500,10.170000,4.761905,0.508500,4.000000
25%,33.202500,3.000000,5.864625,117.292500,4.761905,5.864625,5.500000
50%,54.890000,5.000000,12.066000,241.320000,4.761905,12.066000,6.900000
75%,77.667500,8.000000,22.428000,448.560000,4.761905,22.428000,8.475000
max,99.960000,10.000000,49.650000,993.000000,4.761905,49.650000,10.000000


In [8]:
for colemn in df.select_dtypes(include=['object']).columns:
    print(f"Column: {colemn}")
    print(df[colemn].value_counts())
    print("\n")

Column: Invoice ID
Invoice ID
518-17-2983    2
779-42-2410    2
190-14-3147    2
408-66-6712    2
679-22-6530    2
              ..
233-67-5758    1
303-96-2227    1
727-02-1313    1
347-56-2442    1
849-09-3807    1
Name: count, Length: 1000, dtype: int64


Column: Branch
Branch
Alex     344
Cairo    338
Giza     330
giza       1
cairo      1
Name: count, dtype: int64


Column: City
City
Yangon       344
Mandalay     339
Naypyitaw    331
Name: count, dtype: int64


Column: Customer type
Customer type
Member    570
Normal    444
Name: count, dtype: int64


Column: Gender
Gender
Female    566
Male      426
F          13
M           9
Name: count, dtype: int64


Column: Product line
Product line
Fashion accessories       181
Food and beverages        177
Electronic accessories    172
Sports and travel         168
Home and lifestyle        160
Health and beauty         156
Name: count, dtype: int64


Column: Sales
Sales
829.08      2
189.0945    2
216.846     2
263.97      2
217.6335    2

C:\Users\HP 640 G5\AppData\Local\Temp\ipykernel_13020\336332867.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for colemn in df.select_dtypes(include=['object']).columns:


In [9]:
### OBSERVATIONS BEFORE DATA CLEANING:
# 1. The dataset contains 1014 rows and 17 columns.
# 2. Missing values were found in the unit price, Sales, and date columns.
# 3. Unit price has 24 missing values, Sales has 20 missing values, and date has 12 missing values.
# 4. The dataset contains 14 duplicate rows.
# 5. Some columns have incorrect data types, such as Sales and Date.
# 6. Inconsistent values were found in Branch, Gender, and Payment columns.
# 7. The Branch columns contains both uppercase and lowercase virsions of some branch names.
# 8. The Gender column contains both full names and abbreviations.
# 9. The Payment column contains inconsistent formats such as Ewallet\E-wallet, Cash\cash, Credit card\credit card.

In [10]:
df = df.rename(columns={
    'Invoice ID': 'invoice_id',
    'Branch': 'branch',
    'City': 'city',
    'Customer type': 'customer_type',
    'Gender': 'gender',
    'Product line': 'product_line',
    'Unit price': 'unit_price',
    'Quantity': 'quantity',
    'Tax 5%': 'tax_5_percent',
    'Sales': 'sales',
    'Date': 'date',
    'Time': 'time',
    'Payment': 'payment',
    'cogs': 'cogs',
    'gross margin percentage': 'gross_margin_percentage',
    'gross income': 'gross_income',
    'Rating': 'rating'
})

In [11]:
df.loc[df['unit_price'].isnull(), 'unit_price'] = (
    df.loc[df['unit_price'].isnull(), 'tax_5_percent']
    / (df.loc[df['unit_price'].isnull(), 'quantity'] * 0.05)
)

In [14]:
df['unit_price'].isnull().sum()

np.int64(0)

In [16]:
df['sales'] = pd.to_numeric(df['sales'], errors='coerce')

In [17]:
df['sales'].isnull().sum()

np.int64(29)

In [18]:
df.loc[df['sales'].isnull(), 'sales'] = (
    df.loc[df['sales'].isnull(), 'unit_price']
    * df.loc[df['sales'].isnull(), 'quantity']
    + df.loc[df['sales'].isnull(), 'tax_5_percent']
)

In [19]:
df['sales'].isnull().sum()

np.int64(0)

In [20]:
df = df.dropna(subset=['date'])

In [21]:
df['date'].isnull().sum()

np.int64(0)

In [22]:
df = df.drop_duplicates()

In [23]:
df.duplicated().sum()

np.int64(0)

In [24]:
df = df.drop(columns=['invoice_id'])

In [25]:
df['date'] = pd.to_datetime(df['date'])

In [26]:
df['time'] = pd.to_datetime(df['time'], format='%I:%M:%S %p').dt.time

In [27]:
df['branch'] = df['branch'].str.capitalize()

In [28]:
df['gender'] = df['gender'].replace({
    'F': 'Female',
    'M': 'Male'
})

In [30]:
df['payment'] = df['payment'].replace({
    'E-wallet': 'Ewallet'
})

In [31]:
df = df.sort_values(by='date')

In [32]:
df['Day'] = df['date'].dt.day
df['Month'] = df['date'].dt.month
df['Year'] = df['date'].dt.year

In [33]:
df['satisfied'] = df['rating'].apply(
    lambda x: 'Satisfied' if x >= 7 else 'Not Satisfied'
)

In [34]:
total_revenue = df['sales'].sum()
total_revenue

np.float64(318041.9235)

In [35]:
revenue_by_city = df.groupby('city')['sales'].sum()
revenue_by_city

city
Mandalay     105266.5320
Naypyitaw    109168.7100
Yangon       103606.6815
Name: sales, dtype: float64

In [36]:
highest_revenue_city = revenue_by_city.idxmax()
highest_revenue_city

'Naypyitaw'

In [37]:
profit_by_branch = df.groupby('branch')['gross_income'].sum()
profit_by_branch

branch
Alex     4933.8415
Cairo    5012.8820
Giza     5198.5100
Name: gross_income, dtype: float64

In [38]:
most_profitable_branch = profit_by_branch.idxmax()
most_profitable_branch

'Giza'

In [39]:
category_performance = df.groupby('product_line')[['sales', 'gross_income']].sum()
category_performance

,sales,gross_income
product_line,,
Electronic accessories,52922.5935,2520.1235
Fashion accessories,54245.0790,2583.2890
Food and beverages,55518.6030,2643.9330
Health and beauty,48368.1030,2303.2430
Home and lifestyle,53089.5330,2528.0730
Sports and travel,53898.0120,2566.5720


In [40]:
top_category = category_performance['sales'].idxmax()
top_category

'Food and beverages'

In [41]:
spending_by_customer = df.groupby('customer_type')['sales'].sum()
spending_by_customer

customer_type
Member    184769.9385
Normal    133271.9850
Name: sales, dtype: float64

In [42]:
top_customer_type = spending_by_customer.idxmax()
top_customer_type

'Member'

In [43]:
transactions_by_payment = df['payment'].value_counts()
transactions_by_payment

payment
Cash           341
Ewallet        338
Credit Card    309
Name: count, dtype: int64

In [44]:
most_popular_payment = transactions_by_payment.idxmax()
most_popular_payment

'Cash'

In [45]:
average_transaction_value = df['sales'].mean()
average_transaction_value

np.float64(321.90478087044534)

In [46]:
satisfaction_by_branch = (
    df.groupby('branch')['satisfied']
    .apply(lambda x: (x == 'Satisfied').mean() * 100)
)

satisfaction_by_branch

branch
Alex     52.694611
Cairo    46.036585
Giza     52.147239
Name: satisfied, dtype: float64

In [47]:
highest_satisfaction_branch = satisfaction_by_branch.idxmax()
highest_satisfaction_branch

'Alex'

In [48]:
sales_by_day = df.groupby('Day')['sales'].sum()
sales_by_day

Day
1      9824.0835
2     12646.7565
3     12399.2295
4      7957.6245
5     12798.6915
6      9551.4090
7     11047.2180
8     12869.3985
9     13660.1430
10     9789.0555
11     9618.3675
12    11614.8165
13     5449.0485
14    13635.3420
15    15127.0245
16     9947.3220
17    10418.6145
18     5569.3365
19    14883.2355
20    11647.3245
21     5663.3535
22     7326.2280
23    12428.8185
24    10829.5950
25    10719.2610
26     8828.1900
27    13398.1680
28     9326.1315
29     6790.3185
30     7045.3215
31     5232.4965
Name: sales, dtype: float64

In [49]:
highest_sales_day = sales_by_day.idxmax()
highest_sales_day

np.int32(15)

In [50]:
sales_by_month = df.groupby('Month')['sales'].sum()
sales_by_month

Month
1    115625.2860
2     94445.3790
3    107971.2585
Name: sales, dtype: float64

In [51]:
highest_sales_month = sales_by_month.idxmax()
highest_sales_month

np.int32(1)

In [52]:
overall_satisfaction = (
    (df['satisfied'] == 'Satisfied').mean() * 100
)

overall_satisfaction

np.float64(50.30364372469636)

In [53]:
df

,branch,city,customer_type,gender,product_line,unit_price,quantity,tax_5_percent,sales,date,time,payment,cogs,gross_margin_percentage,gross_income,rating,Day,Month,Year,satisfied
970,Cairo,Mandalay,Member,Female,Food and beverages,84.63,10,42.315,888.615,2019-01-01,11:36:00,Credit Card,846.30,4.761905,42.315,9.0,1,1,2019,Satisfied
696,Alex,Yangon,Member,Female,Sports and travel,27.04,4,5.408,113.568,2019-01-01,20:26:00,Ewallet,108.16,4.761905,5.408,6.9,1,1,2019,Not Satisfied
567,Alex,Yangon,Normal,Female,Fashion accessories,65.74,9,29.583,621.243,2019-01-01,13:55:00,Cash,591.66,4.761905,29.583,7.7,1,1,2019,Satisfied
17,Alex,Yangon,Member,Female,Sports and travel,72.61,6,21.783,457.443,2019-01-01,10:39:00,Credit Card,435.66,4.761905,21.783,6.9,1,1,2019,Not Satisfied
856,Cairo,Mandalay,Normal,Male,Food and beverages,21.12,8,8.448,177.408,2019-01-01,19:31:00,Cash,168.96,4.761905,8.448,6.3,1,1,2019,Not Satisfied
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
158,Cairo,Mandalay,Member,Male,Health and beauty,97.22,9,43.749,918.729,2019-03-30,14:43:00,Ewallet,874.98,4.761905,43.749,6.0,30,3,2019,Not Satisfied
646,Giza,Naypyitaw,Normal,Male,Health and beauty,70.21,6,21.063,442.323,2019-03-30,14:58:00,Cash,421.26,4.761905,21.063,7.4,30,3,2019,Satisfied
963,Giza,Naypyitaw,Member,Male,Electronic accessories,96.82,3,14.523,304.983,2019-03-30,20:37:00,Cash,290.46,4.761905,14.523,6.7,30,3,2019,Not Satisfied
671,Cairo,Mandalay,Member,Male,Food and beverages,93.40,2,9.340,196.140,2019-03-30,16:34:00,Cash,186.80,4.761905,9.340,5.5,30,3,2019,Not Satisfied
